In [ ]:
!pip install openai

In [ ]:
from openai import OpenAI
client = OpenAI(api_key="")
print(client.models.list().data[0].id)

In [ ]:
from openai import OpenAI
client = OpenAI(api_key="")
print(client.models.list().data[0].id)

gpt-3.5-turbo


In [ ]:
import os
from google.colab import userdata

os.environ['OPEN_API_KEY'] = userdata.get('OPEN_API_KEY')

PDF_PATH = '/content/knee_arthroplasty_mng.pdf'  # or '/content/drive/MyDrive/healthcare-ai-learning/data/knee_arthroplasty_mng.pdf' if you used Drive

assert os.path.exists(PDF_PATH), f"PDF not found at {PDF_PATH}"
assert os.environ.get('OPEN_API_KEY'), "OPEN_API_KEY not loaded"
print(f"PDF: {os.path.getsize(PDF_PATH):,} bytes")
print("API key loaded")

PDF: 121,869 bytes
API key loaded


In [ ]:
from google.colab import userdata
userdata.get('OPEN_API_KEY')

In [ ]:
!pip install openai -q

from openai import OpenAI

client = OpenAI()  # picks up OPENAI_API_KEY from env

# Create the vector store
vector_store = client.vector_stores.create(name="masshealth_knee_arthroplasty")
print(f"Vector store created: {vector_store.id}")

# Upload the PDF
with open(PDF_PATH, 'rb') as f:
    file_obj = client.files.create(file=f, purpose="assistants")
print(f"File uploaded:        {file_obj.id}")

# Attach file to vector store
attach = client.vector_stores.files.create(
    vector_store_id=vector_store.id,
    file_id=file_obj.id,
)
print(f"Attached, status:     {attach.status}")

Vector store created: vs_6a37c333178c8191a78b5a03906fa39a
File uploaded:        file-7w3TrUumS1aHSy9ZkVBhoX
Attached, status:     in_progress


In [ ]:
import os
from google.colab import userdata

# Pull from YOUR secret name, store under the env var OpenAI expects
os.environ['OPENAI_API_KEY'] = userdata.get('OPEN_API_KEY')

assert os.environ.get('OPENAI_API_KEY'), "Bridge failed — check secret name and Notebook access toggle"
print("Bridge set: OPEN_API_KEY (secret) → OPENAI_API_KEY (env var)")

Bridge set: OPEN_API_KEY (secret) → OPENAI_API_KEY (env var)


In [ ]:
import time

while True:
    f = client.vector_stores.files.retrieve(
        vector_store_id=vector_store.id,
        file_id=file_obj.id,
    )
    print(f"Status: {f.status}")
    if f.status == "completed":
        break
    if f.status == "failed":
        raise RuntimeError(f"Processing failed: {f.last_error}")
    time.sleep(2)

print(f"\nReady to query. Vector store: {vector_store.id}")

Status: completed

Ready to query. Vector store: vs_6a37c333178c8191a78b5a03906fa39a


In [14]:
query = "What are the three types of knee arthroplasty covered by these MassHealth guidelines?"

response = client.responses.create(
    input=query,
    model="gpt-4o-mini",
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store.id],
    }],
)

print(f"Output items: {[item.type for item in response.output]}\n")

for item in response.output:
    if item.type == "message":
        print("ANSWER:")
        print(item.content[0].text)
        print()
        annotations = item.content[0].annotations
        if annotations:
            files = set(a.filename for a in annotations)
            print(f"Sources cited: {files}")
        else:
            print("No annotations returned — model may have answered from its own knowledge.")

Output items: ['file_search_call', 'message']

ANSWER:
The three types of knee arthroplasty covered by the MassHealth guidelines are:

1. **Total Knee Arthroplasty (TKA)** - This involves the surgical reconstruction or replacement of all surfaces of the knee joint due to various conditions such as rheumatoid arthritis or previous surgical complications.

2. **Partial/Unicompartmental Knee Arthroplasty (UKA/PKA)** - This procedure reconstructs either the medial or lateral compartments of the knee joint, primarily used for patients with isolated compartment disease.

3. **Revision Knee Arthroplasty** - This is performed when a previous knee arthroplasty has failed or resulted in complications.

Sources cited: {'knee_arthroplasty_mng.pdf'}


In [15]:
test_questions = [
    "What is the BMI restriction for unicompartmental knee arthroplasty (UKA), and what are the exceptions for members above that BMI?",
    "Is robot-assisted total knee arthroplasty (Makoplasty) covered by MassHealth? Explain why or why not.",
    "What are the Kellgren-Lawrence grade requirements for total knee arthroplasty, and what does each grade describe?",
]

results = []

for i, q in enumerate(test_questions, 1):
    print(f"\n{'='*70}")
    print(f"Q{i}: {q}")
    print('='*70)

    response = client.responses.create(
        input=q,
        model="gpt-4o-mini",
        tools=[{
            "type": "file_search",
            "vector_store_ids": [vector_store.id],
        }],
        include=["output[*].file_search_call.search_results"],
    )

    answer, annotations, chunks = None, [], []
    for item in response.output:
        if item.type == "message":
            answer = item.content[0].text
            annotations = item.content[0].annotations or []
        elif item.type == "file_search_call":
            if hasattr(item, "search_results") and item.search_results:
                chunks = item.search_results

    print(f"\nANSWER:\n{answer}\n")
    print(f"CHUNKS RETRIEVED: {len(chunks)}")
    for j, c in enumerate(chunks[:3], 1):
        snippet = (c.content[0].text[:160].replace('\n', ' ')
                   if c.content else "(no content)")
        print(f"  [{j}] score={c.score:.3f}  {snippet}...")
    print(f"\nCITATIONS: {len(annotations)}")

    results.append({
        "question": q,
        "answer": answer,
        "num_chunks": len(chunks),
        "num_citations": len(annotations),
        "top_chunk_score": chunks[0].score if chunks else None,
    })

print(f"\n\n{'='*70}\nTested {len(results)} questions. Block 4 complete.")


Q1: What is the BMI restriction for unicompartmental knee arthroplasty (UKA), and what are the exceptions for members above that BMI?

ANSWER:
The BMI restriction for unicompartmental knee arthroplasty (UKA) is set at a maximum of **40**. For members who are younger than 50 years and those whose BMI exceeds 40, exceptions can be made if they provide documentation confirming completion of at least **24 weeks** of unsuccessful non-operative treatment. This treatment must include the use of assistive devices, therapeutic injections, and a weight reduction program.

CHUNKS RETRIEVED: 0

CITATIONS: 2

Q2: Is robot-assisted total knee arthroplasty (Makoplasty) covered by MassHealth? Explain why or why not.

ANSWER:
Robot-assisted total knee arthroplasty (Makoplasty) is not covered by MassHealth. The reason is that this specific procedure is considered investigational or experimental under MassHealth's guidelines for medical necessity. As a result, it fails to meet the criteria for coverage 

In [16]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_0f4e0c6624164dc0006a37c6947c288195ac7b5f3c60170925",
  "created_at": 1782040212.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-4o-mini-2024-07-18",
  "object": "response",
  "output": [
    {
      "id": "fs_0f4e0c6624164dc0006a37c695298481959b6d98829569e209",
      "queries": [
        "Kellgren-Lawrence grade requirements for total knee arthroplasty",
        "Kellgren-Lawrence grades description"
      ],
      "status": "completed",
      "type": "file_search_call",
      "results": [
        {
          "attributes": {},
          "file_id": "file-7w3TrUumS1aHSy9ZkVBhoX",
          "filename": "knee_arthroplasty_mng.pdf",
          "score": 0.7278,
          "text": "The most recent medical evaluation, including a summary of the medical history and the most \r\nrecent physical exam with emphasis on the orthopedic knee examination and testing specific to \r\nthe patient’s condition;\r\n5. Results of any ra